# Semana 07: Primeiro Acesso ao AWS Learner Lab Sandbox, Criação de Instância EC2 (Amazon Linux 2023) e Docker

## Mão na Massa na Nuvem AWS — Fábrica Virtual Smart N1

Bem-vindos ao ambiente de nuvem da **Amazon Web Services (AWS)**! A partir desta semana, saímos das simulações locais e entramos no console real da AWS através do ambiente **AWS Academy Learner Lab Sandbox**.

Nesta aula prática, utilizaremos o **Amazon Linux 2023 (AL2023)**, o sistema operacional nativo e otimizado da AWS para servidores em nuvem.

### Objetivos da Aula
1. Conhecer a interface e a dinâmica operacional do **AWS Academy Learner Lab**.
2. Acessar o **Console de Gerenciamento da AWS** em um ambiente sandbox com créditos educacionais.
3. Provisionar uma instância **Amazon EC2** executando **Amazon Linux 2023**.
4. Configurar um **Security Group** para controlar o tráfego de rede (portas 22 para SSH e 80 para HTTP).
5. Conectar-se à máquina via **EC2 Instance Connect** (direto no navegador com o usuário `ec2-user`).
6. Instalar o **Docker** via gerenciador nativo `dnf` e executar uma aplicação web conteinerizada da **Fábrica Virtual Smart N1**.
7. Dominar as boas práticas de **gestão de custos** (Stop Instance e End Lab) para preservar seus US$ 100 de crédito ao longo do semestre.

---


## 1. Fundamentos: Amazon Linux 2023 × Ubuntu Server

Uma dúvida muito comum na AWS é: **"Devo usar Ubuntu ou Amazon Linux?"**

O **Amazon Linux 2023 (AL2023)** é a distribuição oficial da AWS (baseada na família Fedora/RPM). Suas principais vantagens no ambiente AWS são:

| Item | Amazon Linux 2023 (AL2023) | Ubuntu Server 22.04 LTS |
| :--- | :--- | :--- |
| **Usuário padrão SSH** | **`ec2-user`** | `ubuntu` |
| **Gerenciador de pacotes** | **`dnf`** (moderno, sucessor do `yum`) | `apt` / `apt-get` |
| **Instalação do Docker** | **Nativo nos repositórios:** `sudo dnf install -y docker` | Requer script externo (`get.docker.com`) ou PPA |
| **AWS CLI v2 integrada** | **Já vem pré-instalada de fábrica** | Precisa ser instalada manualmente |
| **Tempo de boot e memória** | Inicialização ultrarrápida, kernel otimizado para EC2 | Kernel genérico, consumo ligeiramente maior de RAM |
| **Suporte oficial** | Mantido e atualizado diretamente pela equipe da AWS | Mantido pela Canonical |

> **Conclusão:** Para laboratórios práticos na AWS, o **Amazon Linux 2023** é excelente pois a instalação do Docker é direta em 1 comando pelo repositório oficial da AWS, e todas as ferramentas de nuvem já vêm prontas!

---


## 2. O Ambiente Sandbox: AWS Academy Learner Lab

O **Learner Lab** disponibiliza uma conta real da AWS com **US$ 100 de orçamento pré-pago** para cada estudante.

| Característica | Detalhe operacional |
| :--- | :--- |
| **Orçamento** | US$ 100,00 por aluno para o semestre inteiro. |
| **Duração da Sessão** | Máximo de 4 horas contínuas. Pode ser renovado/reiniciado com *Start Lab*. |
| **Região Obrigatória** | **`us-east-1` (N. Virginia)**. Todos os recursos devem ser criados nesta região. |
| **Persistência de Dados** | Ao clicar em *End Lab*, as instâncias sofrem **Stop**, mas os discos EBS, arquivos e containers **NÃO são deletados**. |
| **Credenciais de Acesso** | Chave SSH padrão **`vockey`** pré-configurada no laboratório. |
| **Permissões IAM** | Utilize sempre a **`LabRole`** ou o **`LabInstanceProfile`**. |

> **REGRA DE OURO — ECONOMIA DE CRÉDITOS:**
> Ao encerrar o laboratório, **SEMPRE** pare suas instâncias EC2 (`Stop Instance`) e clique em **End Lab**. Nunca deixe instâncias ligadas sem uso.

---


## 3. Arquitetura da Solução

```text
+-----------------------------------------------------------------------------------------+
| AWS Cloud (Região us-east-1 - N. Virginia)                                              |
|                                                                                         |
|   VPC Padrão (Default VPC)                                                              |
|   +---------------------------------------------------------------------------------+   |
|   | Security Group (sg-smartn1-docker)                                              |   |
|   |   -> Inbound: Porta 22 (SSH)       - Restrito / Browser EC2 Instance Connect     |   |
|   |   -> Inbound: Porta 80 (HTTP)      - 0.0.0.0/0 (Acesso Web Público)             |   |
|   |                                                                                 |   |
|   |   +-------------------------------------------------------------------------+   |   |
|   |   | Instância Amazon EC2 (Amazon Linux 2023 - t2.micro / t3.micro)          |   |   |
|   |   | Usuário padrão: ec2-user | IP Público IPv4: ex. 54.x.y.z                |   |   |
|   |   |                                                                         |   |   |
|   |   |   +-----------------------------------------------------------------+   |   |   |
|   |   |   | Docker Engine (instalado nativamente via dnf)                   |   |   |   |
|   |   |   |   +---------------------------------------------------------+   |   |   |   |
|   |   |   |   | Container: web-smartn1 (Nginx Web Server)               |   |   |   |   |
|   |   |   |   | Port Mapping: Host:80 -> Container:80                   |   |   |   |   |
|   |   |   |   | Página Web: Dashboard Operacional Smart N1               |   |   |   |   |
|   |   |   |   +---------------------------------------------------------+   |   |   |   |
|   |   |   +-----------------------------------------------------------------+   |   |   |
|   |   +-------------------------------------------------------------------------+   |   |
|   +---------------------------------------------------------------------------------+   |
+-----------------------------------------------------------------------------------------+
                                      ^
                                      | Requisição HTTP (Porta 80)
                                      |
                              [ Cliente / Navegador ]
```

---


## 4. Roteiro Prático Passo a Passo

### Etapa 1: Acessar o Portal e Iniciar o Learner Lab

1. Acesse o portal da instituição no **AWS Academy** (Canvas LMS).
2. Abra o curso **AWS Academy Learner Lab**.
3. No menu, clique em **Learner Lab** para abrir o painel da sandbox.
4. No canto superior direito, clique no botão **Start Lab**:
   - O status mudará de vermelho para amarelo (*starting...*).
   - Aguarde de 1 a 3 minutos até que o círculo fique **verde** com a indicação **ready**.
5. Quando estiver verde, clique no link com texto **AWS** (ao lado do círculo verde):
   - Uma nova aba do navegador abrirá diretamente no **Console de Gerenciamento da AWS**.
6. **Atenção:** Confira no canto superior direito do Console AWS se a região selecionada é **N. Virginia (`us-east-1`)**.

---


### Etapa 2: Criar o Security Group (Firewall)

1. No Console AWS, na barra de pesquisa superior, digite **EC2** e pressione Enter.
2. No menu lateral esquerdo, em **Network & Security**, clique em **Security Groups**.
3. Clique no botão laranja **Create security group**.
4. Preencha:
   - **Security group name:** `sg-smartn1-docker`
   - **Description:** `Permite SSH administrativo e HTTP porta 80 para Docker`
   - **VPC:** Mantenha a VPC padrão selecionada (*Default VPC*).
5. Na seção **Inbound rules** (Regras de Entrada), adicione as duas regras:

| Tipo (*Type*) | Protocolo | Intervalo de Portas (*Port range*) | Origem (*Source*) | Descrição |
| :--- | :--- | :--- | :--- | :--- |
| **SSH** | TCP | `22` | **Anywhere-IPv4 (`0.0.0.0/0`)** | Terminal via SSH / Instance Connect |
| **HTTP** | TCP | `80` | **Anywhere-IPv4 (`0.0.0.0/0`)** | Tráfego web para o container Nginx |

6. Clique em **Create security group** no final da página.

---


### Etapa 3: Lançar a Instância Amazon EC2 com Amazon Linux 2023

1. No menu lateral esquerdo do painel EC2, clique em **Instances** e depois no botão laranja **Launch instances**.
2. Configure os parâmetros da máquina:
   - **Name and tags:** `ec2-smartn1-docker-lab`
   - **Application and OS Images (AMI):**
     - Selecione **Amazon Linux** (ícone padrão da AWS).
     - Verifique se a AMI selecionada é a **Amazon Linux 2023 AMI** (64-bit x86, Free tier eligible).
   - **Instance type:**
     - Selecione `t2.micro` ou `t3.micro`.
   - **Key pair (login):**
     - Selecione o par de chaves existente **`vockey`** (fornecido pelo Learner Lab).
   - **Network settings (Configurações de Rede):**
     - Clique no botão **Edit** (à direita).
     - Garanta que **Auto-assign public IP** esteja como **Enable** (Habilitado).
     - Em **Firewall (security groups)**, marque **Select existing security group**.
     - Selecione o grupo que criamos: `sg-smartn1-docker`.
   - **Configure storage:**
     - Mantenha o padrão: `8 GiB gp3` (Root volume).
   - **Advanced details (Detalhes avançados):**
     - Em **IAM instance profile**, selecione **`LabInstanceProfile`** (ou `LabRole`, se disponível).
3. Clique no botão laranja **Launch instance** no painel resumo à direita.
4. Clique em **View all instances** e aguarde até que o **Instance state** passe para **Running**.

---


### Etapa 4: Conectar à Instância via EC2 Instance Connect

No Amazon Linux, o acesso ao console pelo navegador é imediato e dispensa instalação de clientes SSH locais.

1. Na lista de instâncias EC2, selecione a caixa ao lado de `ec2-smartn1-docker-lab`.
2. Copie e guarde o **Public IPv4 address** (ex: `54.210.xx.xx`).
3. Clique no botão superior **Connect**.
4. Na aba **EC2 Instance Connect**, repare no campo **User name**:
   - Para Amazon Linux, o usuário padrão é automaticamente **`ec2-user`** (se estivesse no Ubuntu seria `ubuntu`).
5. Clique em **Connect** (botão laranja inferior).
6. O terminal preto do navegador abrirá com o prompt:
   ```text
   [ec2-user@ip-172-31-xx-xx ~]$
   ```

---


### Etapa 5: Instalar e Iniciar o Docker no Amazon Linux 2023

No Amazon Linux 2023, o Docker faz parte dos repositórios oficiais mantidos pela AWS. Não é necessário baixar scripts de terceiros! Execute no terminal da EC2:

```bash
# 1. Atualizar os pacotes do sistema usando o dnf
sudo dnf update -y

# 2. Instalar o pacote oficial do Docker
sudo dnf install -y docker

# 3. Iniciar o serviço do Docker e habilitá-lo no boot da máquina
sudo systemctl enable --now docker

# 4. Adicionar o usuário 'ec2-user' ao grupo docker (para não precisar usar sudo)
sudo usermod -aG docker ec2-user
```

Para ativar a permissão do grupo na sessão atual sem precisar desconectar do terminal:

```bash
newgrp docker
```

Valide a instalação executando:

```bash
docker version
docker info
```

---


### Etapa 6: Executar o Container Nginx e Publicar na Porta 80

Com o Docker operacional, vamos subir um servidor web **Nginx** em container:

```bash
# Executar o container em segundo plano (-d), reinício automático e porta 80 mapeada
docker run -d --name web-smartn1 --restart always -p 80:80 nginx:alpine

# Confirmar se o container está 'Up'
docker ps
```

Agora vamos criar uma página personalizada para a **Fábrica Virtual Smart N1**:

```bash
cat > index.html <<'EOF'
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Smart N1 - AWS Cloud & Docker</title>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
        body { background: #0f172a; color: #f8fafc; display: flex; justify-content: center; align-items: center; min-height: 100vh; padding: 20px; }
        .card { background: #1e293b; border-radius: 16px; padding: 32px; max-width: 600px; width: 100%; box-shadow: 0 10px 25px rgba(0,0,0,0.5); border: 1px solid #334155; text-align: center; }
        .badge { display: inline-block; background: #059669; color: #ecfdf5; font-size: 13px; font-weight: 700; padding: 4px 12px; border-radius: 9999px; margin-bottom: 16px; text-transform: uppercase; letter-spacing: 0.05em; }
        h1 { font-size: 24px; margin-bottom: 8px; color: #38bdf8; }
        h2 { font-size: 16px; font-weight: 400; color: #94a3b8; margin-bottom: 24px; }
        .metrics { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin-bottom: 24px; text-align: left; }
        .metric-box { background: #0f172a; padding: 14px; border-radius: 8px; border: 1px solid #334155; }
        .metric-label { font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600; }
        .metric-val { font-size: 15px; color: #e2e8f0; font-weight: 600; margin-top: 4px; }
        .footer { font-size: 12px; color: #64748b; border-top: 1px solid #334155; padding-top: 16px; }
    </style>
</head>
<body>
    <div class="card">
        <span class="badge">Planta Conectada Online</span>
        <h1>Fábrica Virtual Smart N1</h1>
        <h2>Semana 07 — Instância AWS EC2 com Amazon Linux 2023</h2>
        <div class="metrics">
            <div class="metric-box">
                <div class="metric-label">Provedor Nuvem</div>
                <div class="metric-val">AWS Learner Lab</div>
            </div>
            <div class="metric-box">
                <div class="metric-label">Região AWS</div>
                <div class="metric-val">us-east-1 (N. Virginia)</div>
            </div>
            <div class="metric-box">
                <div class="metric-label">Sistema Operacional</div>
                <div class="metric-val">Amazon Linux 2023</div>
            </div>
            <div class="metric-box">
                <div class="metric-label">Serviço Web</div>
                <div class="metric-val">Docker Nginx (Porta 80)</div>
            </div>
        </div>
        <div class="footer">
            Computação em Nuvem • Automação e Indústria 4.0
        </div>
    </div>
</body>
</html>
EOF

# Copiar a página HTML para dentro do container
docker cp index.html web-smartn1:/usr/share/nginx/html/index.html

# Testar localmente via curl na própria EC2
curl -I http://localhost:80
```

---


### Etapa 7: Validar o Acesso Público no Navegador Web

1. Obtenha o **Public IPv4 address** da sua EC2 no console AWS.
2. Em uma nova aba do navegador, acesse:
   ```text
   http://SEU_IP_PUBLICO
   ```
   *(Atenção: use obrigatoriamente `http://` e não `https://`)*.
3. O painel da **Fábrica Virtual Smart N1** será carregado diretamente da sua máquina em nuvem!

> **Dica de Diagnóstico (Troubleshooting):**
> - Se o navegador ficar carregando sem parar (timeout), o problema é o **Security Group**: confira se a regra de entrada para HTTP na porta 80 com origem `0.0.0.0/0` está salva.
> - Se receber erro `Connection Refused`, confira se o container está rodando com `docker ps`.

---


### Etapa 8: Procedimento de Final de Aula (Desligamento e Economia de Créditos)

1. No Console AWS, selecione sua instância `ec2-smartn1-docker-lab`.
2. Vá em **Instance state → Stop instance** (Parar instância).
   - **ATENÇÃO:** Nunca selecione *Terminate instance* (isso destruiria a máquina permanentemente). A opção correta é **Stop instance**.
3. Aguarde o status mudar para **Stopped**.
4. Retorne à aba do **AWS Academy Learner Lab**.
5. Clique no botão vermelho **End Lab**.

Na próxima aula, ao clicar em **Start Lab**, basta dar **Start instance** na EC2 e tudo estará intacto no seu disco EBS!

---


## 5. Exercícios de Avaliação e Entregáveis

Para validação da participação e comprovação da prática, envie as 3 capturas de tela abaixo:

### Evidência 1: Console AWS
- Print do painel EC2 exibindo a instância com o sistema operacional **Amazon Linux 2023**, estado **Running**, região `us-east-1` e endereço IP Público visíveis.

### Evidência 2: Terminal do Amazon Linux 2023
- Print da janela do EC2 Instance Connect mostrando a saída de:
  ```bash
  cat /etc/os-release
  docker ps
  ```

### Evidência 3: Aplicação Web em Produção
- Print da aba do navegador aberta em `http://<SEU_IP_PUBLICO>` com a página estilizada da **Fábrica Virtual Smart N1**.

### Pergunta Conceitual
Por que o usuário padrão do Amazon Linux é `ec2-user` e qual a vantagem de instalar o Docker via `dnf` diretamente dos repositórios mantidos pela AWS em comparação com baixar scripts externos de terceiros?